# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession

# 1. Initialisation de la session Spark pour le Jour 3
spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

# Chemin de votre table Silver validée précédemment
SILVER_MAIN_PATH = "hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver"

# Enregistrement de la table Silver sous forme de vue temporaire SQL
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

:: loading settings :: url = jar:file:/opt/python/cpython-3.15.2-linux-aarch64-gnu/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0f3e7f1b-60e9-41a5-92d6-6bda5e45b8fd;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta

Lignes disponibles dans v_silver : 46953366


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [ ]:
print("\n--- Exercice 1.1 : Résumé statistique & Asymétrie ---")
df_dist_summary = spark.sql("""
    SELECT
        COUNT(*) AS total_trips,
        AVG(fare_amount) AS avg_fare,
        STDDEV(fare_amount) AS stddev_fare,
        SKEWNESS(fare_amount) AS skewness_fare,
        PERCENTILE_APPROX(fare_amount, 0.25) AS p25,
        PERCENTILE_APPROX(fare_amount, 0.50) AS mediane,
        PERCENTILE_APPROX(fare_amount, 0.75) AS p75,
        PERCENTILE_APPROX(fare_amount, 0.90) AS p90,
        PERCENTILE_APPROX(fare_amount, 0.95) AS p95,
        MAX(fare_amount) AS max_fare
    FROM v_silver
""")
df_dist_summary.show(truncate=False)


--- Exercice 1.1 : Résumé statistique & Asymétrie ---


+-----------+------------------+-----------------+------------------+---+-------+----+----+----+--------+
|total_trips|avg_fare          |stddev_fare      |skewness_fare     |p25|mediane|p75 |p90 |p95 |max_fare|
+-----------+------------------+-----------------+------------------+---+-------+----+----+----+--------+
|46953366   |12.344307166178186|47.75504240629443|2702.9463638711254|6.5|9.0    |14.0|23.0|32.5|999.98  |
+-----------+------------------+-----------------+------------------+---+-------+----+----+----+--------+



**Votre justification statistique *(À compléter)* :**
> `Expliquez ici ce que révèle le coefficient d'asymétrie (skewness) et l'écart entre la moyenne et la médiane sur vos données.`
Écart moyenne vs médiane : La moyenne arithmétique (AVG) est sensiblement supérieure à la médiane (P50), car elle est tirée vers le haut par les valeurs extrêmes (P90, P95, MAX). La médiane reflète donc beaucoup mieux le montant d'une course
---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [8]:
print("\n--- Exercice 2.1 : Construction de la Table Gold #1 (Pareto) ---")
GOLD_PARETO_PATH = "hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_pareto"

df_gold_1 = spark.sql("""
    WITH payment_stats AS (
        SELECT
            payment_type_name,
            COUNT(*) AS volume_trips,
            SUM(fare_amount) AS total_revenue,
            AVG(fare_amount) AS avg_revenue,
            PERCENTILE_APPROX(fare_amount, 0.50) AS median_revenue,
            (PERCENTILE_APPROX(fare_amount, 0.75) - PERCENTILE_APPROX(fare_amount, 0.25)) AS iqr_revenue
        FROM v_silver
        WHERE payment_type_name IS NOT NULL
        GROUP BY payment_type_name
    )
    SELECT
        payment_type_name,
        volume_trips,
        total_revenue,
        avg_revenue,
        median_revenue,
        iqr_revenue,
        DENSE_RANK() OVER (ORDER BY total_revenue DESC) as rank_revenue,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(total_revenue) OVER () AS cumulated_pareto_ratio
    FROM payment_stats
""")

# Sauvegarde au format Delta avec coalesce(1) pour éviter les petits fichiers
(
    df_gold_1
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_PARETO_PATH)
)

print(f"Table Gold #1 sauvegardée avec succès dans {GOLD_PARETO_PATH}")
df_gold_1.show(15, truncate=False)


--- Exercice 2.1 : Construction de la Table Gold #1 (Pareto) ---


26/11/19 08:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:40:42 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:41:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

Table Gold #1 sauvegardée avec succès dans hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_pareto


26/11/19 08:41:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:41:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:41:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:41:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:41:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 0

+-----------------+------------+--------------------+------------------+--------------+-----------+------------+----------------------+
|payment_type_name|volume_trips|total_revenue       |avg_revenue       |median_revenue|iqr_revenue|rank_revenue|cumulated_pareto_ratio|
+-----------------+------------+--------------------+------------------+--------------+-----------+------------+----------------------+
|Credit card      |30767661    |3.9456603855998087E8|12.824050504195975|9.5           |8.0        |1           |0.6807478058494845    |
|Cash             |16017089    |1.8240959110999978E8|11.388435883074619|8.5           |7.0        |2           |0.9954604693296618    |
|No charge        |124790      |2005114.9399999997  |16.067913614872985|8.5           |12.0       |3           |0.9989199094803213    |
|Dispute          |43823       |626011.78           |14.285005134290213|9.5           |11.5       |4           |0.9999999723950775    |
|Unknown          |3           |16.0            

26/11/19 08:42:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:42:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:42:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 08:42:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [ ]:
GOLD_TIME_PATH = "hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_time"

df_gold_2 = spark.sql("""
    WITH monthly_revenue AS (
        SELECT
            trip_month,
            COUNT(*) AS total_trips,
            SUM(fare_amount) AS total_revenue,
            AVG(fare_amount) AS avg_fare
        FROM v_silver
        WHERE trip_month IS NOT NULL
        GROUP BY trip_month
    )
    SELECT
        trip_month,
        total_trips,
        total_revenue,
        avg_fare,
        LAG(total_revenue, 1) OVER (ORDER BY trip_month) AS prev_month_revenue,
        (total_revenue - LAG(total_revenue, 1) OVER (ORDER BY trip_month)) AS revenue_diff,
        ROUND(
            ((total_revenue - LAG(total_revenue, 1) OVER (ORDER BY trip_month)) / LAG(total_revenue, 1) OVER (ORDER BY trip_month)) * 100,
            2
        ) AS growth_rate_pct
    FROM monthly_revenue
    ORDER BY trip_month
""")

# Sauvegarde au format Delta avec .coalesce(1)
(
    df_gold_2
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TIME_PATH)
)

print(f"Table Gold #2 sauvegardée avec succès dans {GOLD_TIME_PATH}")
df_gold_2.show(20, truncate=False)

26/11/18 13:48:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Table Gold #2 sauvegardée avec succès dans hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_time


26/11/18 13:48:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+----------+-----------+--------------------+------------------+--------------------+--------------------+---------------+
|trip_month|total_trips|total_revenue       |avg_fare          |prev_month_revenue  |revenue_diff        |growth_rate_pct|
+----------+-----------+--------------------+------------------+--------------------+--------------------+---------------+
|2015-01   |12666557   |1.5031724719998857E8|11.867253840170504|NULL                |NULL                |NULL           |
|2016-01   |10839545   |1.3488305582999992E8|12.443608641322115|1.5031724719998857E8|-1.543419136998865E7|-10.27         |
|2016-02   |11311030   |1.3998853198000002E8|12.376285093399984|1.3488305582999992E8|5105476.150000095   |3.79           |
|2016-03   |12136234   |1.5441793738000014E8|12.723711274848537|1.3998853198000002E8|1.4429405400000125E7|10.31          |
+----------+-----------+--------------------+------------------+--------------------+--------------------+---------------+



26/11/18 13:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/18 13:48:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [7]:
GOLD_TOP_PATH = "hdfs://namenode:9000/lakehouse/gold/nyc_taxi_gold_top"

# 1. Création de la 3e table Gold (Top course par type de paiement par mois)
df_gold_3 = spark.sql("""
    WITH ranked_trips AS (
        SELECT
            trip_month,
            payment_type_name,
            trip_distance,
            fare_amount,
            ROW_NUMBER() OVER (PARTITION BY trip_month, payment_type_name ORDER BY fare_amount DESC) as rn
        FROM v_silver
        WHERE trip_month IS NOT NULL AND payment_type_name IS NOT NULL
    )
    SELECT
        trip_month,
        payment_type_name,
        trip_distance,
        fare_amount
    FROM ranked_trips
    WHERE rn = 1
""")

(
    df_gold_3
    .coalesce(1)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(GOLD_TOP_PATH)
)
print("Table Gold #3 créée et sauvegardée avec succès.")

# 2. Compaction de la table Silver (Résout le Small Files Problem)
print("Lancement de OPTIMIZE sur la couche Silver...")
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
print("Compaction terminée avec succès ! 🎉")

Table Gold #3 créée et sauvegardée avec succès.
Lancement de OPTIMIZE sur la couche Silver...


+-----------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                 |metrics                                                                                                                                                                                                        |
+-----------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|hdfs://namenode:9000/lakehouse/silver/nyc_taxi_silver|{2, 34, {958890877, 966583356, 9.627371165E8, 2, 1925474233}, {13042889, 490612355, 5.399110602941176E7, 34, 1835697605}, 1, NULL, NULL, 2, 1, 34, 0, fal